In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [3]:
#Merge(Join) 2 dataframes based on columns
ds = pd.merge(movies, credits, left_on='id', right_on='movie_id', how='inner')

In [4]:
ds = ds[['id', 'title_x', 'genres', 'keywords', 'overview', 'production_companies', 
         'tagline', 'cast', 'crew']]

In [5]:
ds.rename(columns={'title_x': 'title'}, inplace=True)

In [7]:
#Function which receives string of list of dict 
#converts to list and iterate all dict and returns comma seperated list of 'name' from dict
import ast
def get_names(mystring):
    genres = ast.literal_eval(mystring)
    words = []
    for g in genres:
        words.append(g["name"])
    words =  ','.join(words)
    return words

In [8]:
ds["genres"] = ds["genres"].apply(get_names)

In [9]:
ds["keywords"] = ds["keywords"].apply(get_names)

In [11]:
ds["production_companies"] = ds["production_companies"].apply(get_names)

In [14]:
ds["cast"] = ds["cast"].apply(get_names)

In [15]:
ds["crew"] = ds["crew"].apply(get_names)

In [25]:
def replace_spaces_with_comma(sentence):
    sentence = str(sentence)
    sentence = sentence.replace(",", "")
    sentence = sentence.replace(" ", ",")
    return sentence

In [26]:
ds["overview"] = ds["overview"].apply(replace_spaces_with_comma)
ds["tagline"] = ds["tagline"].apply(replace_spaces_with_comma)

In [36]:
ds["tags"] = ds['genres'] + "," + ds['keywords'] + "," + ds['overview'] + "," + ds['production_companies'] + "," + ds['tagline'] + "," + ds['cast'] + "," + ds['crew']

In [40]:
ds["tags"] = ds["tags"].apply(lambda val:val.lower())

In [42]:
ds = ds[['id', 'title', 'tags']]

In [44]:
def get_unique_words(original_string):
    # Step 1: Split, strip, and remove duplicates using set
    words = [word.strip() for word in original_string.split(',')]
    unique_words = list(set(words))
    # Optional: sort if you want consistent output
    unique_words.sort()
    # Step 2: Join back into a comma-separated string
    new_string = ','.join(unique_words)
    return new_string

In [47]:
ds["tags"] = ds["tags"].apply(get_unique_words)

In [50]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Abhijit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [52]:
def remove_stop_words(text):
    # Get the English stopwords list
    stop_words = set(stopwords.words('english'))
    # Tokenize the sentence
    words = text.split(',')
    # Remove stopwords
    filtered_words = [word for word in words if word.lower() not in stop_words]
    # # Convert back to string
    filtered_text = ",".join(filtered_words)
    return filtered_text

In [55]:
ds["tags"] =ds["tags"].apply(remove_stop_words)

In [58]:
ds["tags"] = ds["tags"].apply(lambda val:val.replace(" ", "_"))

In [60]:
ds.to_csv("cleaned_data.csv")

In [61]:
ds.shape

(4803, 3)